# Phase 1 (v2) — ML emulator of FLEXPART's real convection scheme

This version trains on **FLEXPART's own convection code** (Emanuel & Živković-Rothman scheme,
`CONVECT` + `TLIFT` from `conv_mod.f90`, FLEXPART 11), compiled as a standalone library.

**Pipeline**

1. Download FLEXPART, extract `CONVECT` + `TLIFT`, patch the `ERROR STOP`, compile (Section 1).
2. Atmospheric columns on 32 levels, 1000–10 hPa — synthetic by default, or ERA5.
3. Real FLEXPART scheme → redistribution matrix `M[i, j]` (fraction of air moved from level *i*
   to level *j*) and the new cloud-base mass flux `cbmf_new`.
4. Three small models:
   - **Gate:** is convection active?
   - **Redistribution net:** predicts `M` (row-wise softmax → every row sums to 1).
   - **Cbmf net:** predicts `cbmf_new` (the scheme has memory: `cbmf_old` is an input).
5. Validation, then TorchScript export for Phase 2 (coupling into FLEXPART via FTorch).

**How to run:** `Runtime → Run all`. A GPU runtime is optional.
If the FLEXPART build fails, the notebook falls back to the simplified stand-in scheme and says so.

## 0. Setup

In [ ]:
import os, re, time, json, ctypes, subprocess, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from numpy.ctypeslib import ndpointer
warnings.filterwarnings("ignore", category=FutureWarning, module="torch.jit")  # TorchScript is what FTorch loads

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CFG = {
    "teacher": os.environ.get("TEACHER", "flexpart"),  # "flexpart" (real scheme) or "standin"
    "data_source": "synthetic",    # "synthetic" or "era5"
    "n_columns": int(os.environ.get("N_COLUMNS", 60000)),
    "delt": 900.0,                 # convection time step (s), FLEXPART's usual synchronisation interval
    "max_spinup_calls": 8,         # random spin-up calls per column → realistic spread of cbmf_old
    "mount_drive": True,
    "seed": 42,
    "gate_epochs": int(os.environ.get("GATE_EPOCHS", 15)),
    "redist_epochs": int(os.environ.get("REDIST_EPOCHS", 40)),
    "cbmf_epochs": int(os.environ.get("CBMF_EPOCHS", 25)),
    "batch_size": 256,
    "lr": 1e-3,
}

if IN_COLAB and CFG["mount_drive"]:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/flexpart_phase1_v2"
else:
    OUT_DIR = os.path.abspath("flexpart_phase1_v2_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
rng = np.random.default_rng(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Colab: {IN_COLAB} | device: {DEVICE} | outputs: {OUT_DIR}")

# 27 ERA5 pressure levels + 5 stratospheric levels: the scheme needs room above the tropopause
P_LEVELS = np.array([1000, 975, 950, 925, 900, 875, 850, 825, 800, 775, 750,
                     700, 650, 600, 550, 500, 450, 400, 350, 300, 250,
                     225, 200, 175, 150, 125, 100, 70, 50, 30, 20, 10], dtype=np.float64)
NLEV = len(P_LEVELS)
PH_LEVELS = np.concatenate([[P_LEVELS[0] + (P_LEVELS[0] - P_LEVELS[1]) / 2],
                            (P_LEVELS[:-1] + P_LEVELS[1:]) / 2, [P_LEVELS[-1] / 2]])  # half levels (hPa)
LAYER_DP = PH_LEVELS[:-1] - PH_LEVELS[1:]
LAYER_W = LAYER_DP / LAYER_DP.sum()
print(f"{NLEV} levels, {P_LEVELS[0]:.0f}–{P_LEVELS[-1]:.0f} hPa")

## 1. Build FLEXPART's convection scheme as a standalone library

- Clones FLEXPART and copies `CONVECT` + `TLIFT` out of `conv_mod.f90`.
- Replaces the `ERROR STOP` ("Convection reaches top level of input data") with a flag, so an
  unusual column returns `iflag = -1` instead of crashing the session. FLEXPART's own files are untouched.
- Wraps one column in a function that mirrors FLEXPART's `calcmatrix`, converting the mass matrix
  to fractions (each row divided by the layer's air mass, so rows sum to 1).

In [ ]:
FLEXPART_REPO = "https://gitlab.phaidra.org/flexpart/flexpart.git"
SRC, BUILD = "flexpart/src", "build"
LIB_PATH = os.path.join(BUILD, "libflexconv.so")

MODULE_HEAD = """module conv_standalone_mod
  use par_mod
  implicit none
  integer :: nconvlevmax = 0, na = 0, nconvlev = 0, nconvtop = 0, hit_top = 0
  real, allocatable, dimension(:,:) :: pconv, phconv, dpr, pconv_hpa, phconv_hpa, &
                                        sub, tconv, qconv, qsconv
  real, allocatable, dimension(:,:,:) :: fmass, fmassfrac
  real :: psconv, tt2conv, td2conv
contains
"""

WRAPPER = """module flexconv_wrap
  use iso_c_binding
  use qvsat_mod, only: f_qvsat
  use conv_standalone_mod
  implicit none
contains
  subroutine column_matrix(nlev, t, q, p_pa, ph_pa, delt, cbmf_in, frac, cbmf_out, iflag, ntop) &
      bind(C, name="column_matrix")
    integer(c_int), value :: nlev
    real(c_double), intent(in) :: t(nlev), q(nlev), p_pa(nlev), ph_pa(nlev+1)
    real(c_double), value :: delt, cbmf_in
    real(c_double), intent(out) :: frac(nlev, nlev), cbmf_out
    integer(c_int), intent(out) :: iflag, ntop
    real, parameter :: g = 9.81
    real :: precip, wd, tprime, qprime, cbmf, rdelt, rlevmass, summe, row(nlev)
    integer :: k, ifl

    if (nconvlevmax /= nlev) then
      if (allocated(tconv)) deallocate(pconv, phconv, dpr, pconv_hpa, phconv_hpa, sub, &
                                       tconv, qconv, qsconv, fmass, fmassfrac)
      nconvlevmax = nlev; na = nlev + 1
      allocate(pconv(nconvlevmax,1), phconv(na,1), dpr(nconvlevmax,1), &
               pconv_hpa(nconvlevmax,1), phconv_hpa(na,1), sub(nconvlevmax,1), &
               tconv(na,1), qconv(na,1), qsconv(na,1), &
               fmass(nconvlevmax,nconvlevmax,1), fmassfrac(nconvlevmax,nconvlevmax,1))
    end if
    nconvlev = nlev - 1

    tconv = 0.; qconv = 0.; qsconv = 0.; fmass = 0.; sub = 0.
    tconv(1:nlev,1) = real(t); qconv(1:nlev,1) = real(q)
    tconv(na,1) = real(t(nlev)); qconv(na,1) = real(q(nlev))
    pconv(:,1) = real(p_pa); phconv(:,1) = real(ph_pa)
    psconv = real(ph_pa(1))
    do k = 1, nlev
      qsconv(k,1) = f_qvsat(pconv(k,1), tconv(k,1))
      dpr(k,1) = phconv(k,1) - phconv(k+1,1)
    end do
    qsconv(na,1) = qsconv(nlev,1)
    pconv_hpa(:,1) = pconv(:,1) / 100.; phconv_hpa(:,1) = phconv(:,1) / 100.

    rdelt = real(delt); cbmf = real(cbmf_in); hit_top = 0
    call convect(nconvlevmax, nconvlev, rdelt, ifl, precip, wd, tprime, qprime, cbmf, 1)
    iflag = ifl

    frac = 0.d0
    do k = 1, nlev
      frac(k,k) = 1.d0
    end do
    ntop = 0
    cbmf_out = cbmf
    if (hit_top == 1) then
      iflag = -1; cbmf_out = cbmf_in
      return
    end if
    if ((ifl /= 1 .and. ifl /= 4) .or. (cbmf <= 0. .and. real(cbmf_in) <= 0.)) then
      cbmf_out = cbmf_in
      return
    end if
    ntop = nconvtop
    do k = 1, nconvtop
      rlevmass = dpr(k,1) / g
      row(1:nconvtop) = rdelt * fmass(k,1:nconvtop,1)
      summe = sum(row(1:nconvtop))
      row(k) = row(k) + rlevmass - summe
      frac(k,1:nconvtop) = row(1:nconvtop) / rlevmass
    end do
  end subroutine column_matrix
end module flexconv_wrap
"""

def build_flexpart_library():
    if not os.path.exists(f"{SRC}/conv_mod.f90"):
        r = subprocess.run(f"git clone -q {FLEXPART_REPO}", shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print("Could not clone FLEXPART:", r.stderr[-500:]); return False
    os.makedirs(BUILD, exist_ok=True)
    lines = open(f"{SRC}/conv_mod.f90").read().splitlines()
    start = next(i for i, l in enumerate(lines) if re.match(r"\s*SUBROUTINE\s+CONVECT\b", l, re.I))
    end = next(i for i, l in enumerate(lines) if re.match(r"\s*END\s+SUBROUTINE\s+TLIFT\b", l, re.I))
    body = "\n".join(lines[start:end + 1])
    body, n = re.subn(r"error\s+stop\s+'Convection reaches top level of input data\.\s*&\s*\n\s*More levels are needed\.'",
                      "nconvtop = nconvlevmax\n    hit_top = 1", body, flags=re.I)
    if n != 1:
        print("Warning: ERROR STOP not found — the FLEXPART version may differ.")
    open(f"{BUILD}/conv_standalone_mod.f90", "w").write(MODULE_HEAD + body + "\nend module conv_standalone_mod\n")
    open(f"{BUILD}/flexconv_wrap.f90", "w").write(WRAPPER)
    if subprocess.run("which gfortran", shell=True, capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y gfortran", shell=True)
    cmd = (f"gfortran -O2 -fPIC -shared -cpp -ffree-line-length-none -J{BUILD} "
           f"{SRC}/par_mod.f90 {SRC}/qvsat_mod.f90 {BUILD}/conv_standalone_mod.f90 "
           f"{BUILD}/flexconv_wrap.f90 -o {LIB_PATH}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("Compile failed:\n", r.stderr[-2000:]); return False
    return True

FLEXPART_OK = False
if CFG["teacher"] == "flexpart":
    FLEXPART_OK = os.path.exists(LIB_PATH) or build_flexpart_library()
    if FLEXPART_OK:
        _lib = ctypes.CDLL(os.path.abspath(LIB_PATH))
        _arr = lambda: ndpointer(dtype=np.float64, flags="F_CONTIGUOUS")
        _lib.column_matrix.argtypes = [ctypes.c_int, _arr(), _arr(), _arr(), _arr(), ctypes.c_double,
                                       ctypes.c_double, _arr(), ctypes.POINTER(ctypes.c_double),
                                       ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int)]
        _lib.column_matrix.restype = None
        print("FLEXPART convection library ready:", LIB_PATH)
    else:
        print("⚠️ FLEXPART build failed — falling back to the STAND-IN scheme.")
        CFG["teacher"] = "standin"

_P_PA = np.asfortranarray(P_LEVELS * 100.0)
_PH_PA = np.asfortranarray(PH_LEVELS * 100.0)

def flexpart_column(T, q, cbmf_in, delt=CFG["delt"]):
    """One call of FLEXPART's convection for one column → (M, cbmf_new, iflag, ntop)."""
    frac = np.zeros((NLEV, NLEV), order="F")
    c, f, t = ctypes.c_double(), ctypes.c_int(), ctypes.c_int()
    _lib.column_matrix(NLEV, np.asfortranarray(T, dtype=np.float64), np.asfortranarray(q, dtype=np.float64),
                       _P_PA, _PH_PA, delt, cbmf_in, frac, ctypes.byref(c), ctypes.byref(f), ctypes.byref(t))
    return frac, c.value, f.value, t.value

## 2. Thermodynamics helpers (input features: CAPE, CIN, LCL)

In [ ]:
RD, RV, CP, LV, G = 287.04, 461.5, 1004.6, 2.501e6, 9.81
EPS = RD / RV

def sat_vapour_pressure(T):
    return 611.2 * np.exp(17.67 * (T - 273.15) / (T - 29.65))

def q_sat(T, p_pa):
    es = sat_vapour_pressure(T)
    return EPS * es / (p_pa - (1 - EPS) * es)

def r_sat(T, p_pa):
    es = sat_vapour_pressure(T)
    return EPS * es / np.maximum(p_pa - es, 1.0)

def virtual_temp(T, q):
    return T * (1 + 0.608 * q)

def parcel_diagnostics(T, q, p_hpa=P_LEVELS, nsub=10):
    """Surface-parcel ascent → CAPE, CIN, LCL index/pressure, LNB index (vectorised)."""
    N, n = T.shape
    p = p_hpa * 100.0
    Tp = np.empty_like(T); Tp[:, 0] = T[:, 0]
    q0 = q[:, 0].copy(); Tcur = T[:, 0].copy()
    sat = q0 >= q_sat(Tcur, p[0])
    lcl_idx = np.where(sat, 0, n - 1)
    for k in range(1, n):
        p_prev = p[k - 1]
        for s in range(1, nsub + 1):
            p_new = p[k - 1] * (p[k] / p[k - 1]) ** (s / nsub)
            dp = p_new - p_prev
            T_dry = Tcur * (p_new / p_prev) ** (RD / CP)
            rs = r_sat(Tcur, p_prev)
            dTdp = (RD * Tcur + LV * rs) / (p_prev * (CP + LV**2 * rs * EPS / (RD * Tcur**2)))
            Tcur = np.where(sat, Tcur + dTdp * dp, T_dry)
            newly = (~sat) & (q0 >= q_sat(Tcur, p_new))
            lcl_idx = np.where(newly & (lcl_idx == n - 1), k, lcl_idx)
            sat = sat | newly
            p_prev = p_new
        Tp[:, k] = Tcur
    qp = np.where(np.arange(n)[None, :] >= lcl_idx[:, None], q_sat(Tp, p[None, :]), q0[:, None])
    B = virtual_temp(Tp, qp) - virtual_temp(T, q)
    dlnp = np.log(p[:-1] / p[1:])
    layer = RD * 0.5 * (B[:, :-1] + B[:, 1:]) * dlnp[None, :]
    cape = np.clip(layer, 0, None).sum(axis=1)
    pos = layer > 0
    first_pos = np.where(pos.any(axis=1), pos.argmax(axis=1), n - 1)
    cin = np.where(np.arange(n - 1)[None, :] < first_pos[:, None], np.clip(layer, None, 0), 0).sum(axis=1)
    has_pos = B > 0
    lnb_idx = np.where(has_pos.any(axis=1), n - 1 - has_pos[:, ::-1].argmax(axis=1), -1)
    return {"cape": cape, "cin": cin, "lcl_idx": lcl_idx, "lcl_p": p_hpa[lcl_idx], "lnb_idx": lnb_idx}

## 3. Atmospheric columns
Synthetic soundings now include a stratosphere (temperature rising above the tropopause), which
the scheme needs to stop rising air. The ERA5 loader needs extra stratospheric levels for real use
(ERA5 pressure levels go to 1 hPa; request them, or use model levels).

In [ ]:
REGIMES = {  # probability, surface T (K), surface RH, lapse rate (K/km)
    "tropical_ocean": (0.35, (298, 304), (0.75, 0.95), (5.0, 6.5)),
    "monsoon_land":   (0.25, (300, 310), (0.50, 0.90), (6.0, 8.0)),
    "midlatitude":    (0.25, (275, 295), (0.50, 0.90), (5.5, 7.5)),
    "dry":            (0.15, (290, 312), (0.10, 0.40), (7.0, 9.5)),
}

def synthetic_columns(N, rng):
    names = list(REGIMES)
    probs = np.array([REGIMES[k][0] for k in names])
    reg = rng.choice(len(names), size=N, p=probs / probs.sum())
    Ts, RHs, gam = np.empty(N), np.empty(N), np.empty(N)
    for i, k in enumerate(names):
        m = reg == i
        _, tr, rr, gr = REGIMES[k]
        Ts[m] = rng.uniform(*tr, m.sum()); RHs[m] = rng.uniform(*rr, m.sum()); gam[m] = rng.uniform(*gr, m.sum())
    p = P_LEVELS[None, :]
    kexp = (RD * (gam / 1000.0) / G)[:, None]
    T_trop = rng.uniform(190, 215, size=N)[:, None]
    warm = rng.uniform(12, 25, size=N)[:, None]          # stratospheric warming per ln(p)
    p_trop = 1000.0 * (T_trop / Ts[:, None]) ** (1.0 / kexp)
    T = np.where(p >= p_trop, Ts[:, None] * (p / 1000.0) ** kexp, T_trop + warm * np.log(p_trop / p))
    T = T + rng.normal(0, 0.5, size=T.shape)
    H_rh = rng.uniform(250, 600, size=N)[:, None]
    A = rng.uniform(-0.3, 0.3, size=N)[:, None]
    pc = rng.uniform(400, 750, size=N)[:, None]
    RH = np.clip(RHs[:, None] * np.exp(-(1000.0 - p) / H_rh) + A * np.exp(-((p - pc) / 80.0) ** 2), 0.02, 1.0)
    q = RH * q_sat(T, p * 100.0)
    ps = rng.uniform(1000, 1020, size=N)
    return T, q, ps, reg

def load_era5_columns(path):
    """ERA5 pressure-level file with T ('t') and q ('q') on all P_LEVELS (untested; check names on first use)."""
    import xarray as xr
    ds = xr.open_dataset(path); print(ds)
    lev = "pressure_level" if "pressure_level" in ds.dims else "level"
    tdim = "valid_time" if "valid_time" in ds.dims else "time"
    ds = ds.sel({lev: P_LEVELS})
    T = ds["t"].transpose(tdim, "latitude", "longitude", lev).values.reshape(-1, NLEV)
    q = ds["q"].transpose(tdim, "latitude", "longitude", lev).values.reshape(-1, NLEV)
    times = np.repeat(ds[tdim].values, ds.sizes["latitude"] * ds.sizes["longitude"])
    return T.astype(np.float64), q.astype(np.float64), np.full(len(T), 1013.0), times

if CFG["data_source"] == "era5":
    T_all, q_all, ps_all, times_all = load_era5_columns(os.path.join(OUT_DIR, "era5_columns.nc"))
else:
    T_all, q_all, ps_all, _ = synthetic_columns(CFG["n_columns"], rng)
    times_all = None
print(f"Columns: {len(T_all):,} × {NLEV} levels")

## 4. Teacher labels

**Real FLEXPART scheme (default).** The scheme has memory: the cloud-base mass flux `cbmf` from
the previous time step is an input. Each column gets a random number of spin-up calls (0–8),
which gives a realistic spread of `cbmf_old`; the next call provides the label
(`M`, `cbmf_new`). Columns returning `iflag = -1` (too short) are dropped.

**Stand-in** (fallback): the simplified scheme from v1, with no memory (`cbmf` = 0).

In [ ]:
def teacher_flexpart(T, q):
    N = len(T)
    M = np.empty((N, NLEV, NLEV)); cb_old = np.zeros(N); cb_new = np.zeros(N)
    iflag = np.zeros(N, dtype=int); ntop = np.zeros(N, dtype=int)
    n_spin = rng.integers(0, CFG["max_spinup_calls"] + 1, size=N)
    for c in range(N):
        cb = 0.0
        for _ in range(n_spin[c]):
            _, cb, fl, _ = flexpart_column(T[c], q[c], cb)
            if fl == -1:
                break
        M[c], cb_new[c], iflag[c], ntop[c] = flexpart_column(T[c], q[c], cb)
        cb_old[c] = cb
        if c and c % 20000 == 0:
            print(f"  labelled {c:,} / {N:,}")
    conv = np.isin(iflag, [1, 4]) & (ntop > 0)
    valid = iflag != -1
    return M, conv, cb_old, cb_new, iflag, valid

def teacher_standin(T, q):
    diag = parcel_diagnostics(T, q)
    N = len(T)
    lcl, lnb, cape, cin = diag["lcl_idx"], diag["lnb_idx"], diag["cape"], diag["cin"]
    conv = (cape > 300) & (cin > -50) & (lnb >= lcl + 3)
    M = np.broadcast_to(np.eye(NLEV), (N, NLEV, NLEV)).copy()
    lev = np.arange(NLEV)
    for c in np.where(conv)[0]:
        L, T_ = lcl[c], lnb[c]
        f = np.clip(cape[c] / 4000.0, 0.05, 0.6)
        cloud = (lev > L) & (lev <= T_)
        gauss = np.exp(-0.5 * ((lev - (T_ - 1)) / 2.0) ** 2) * cloud
        d = 0.7 * gauss / gauss.sum() + 0.3 * cloud / cloud.sum()
        src = lev <= L
        s = min(0.5, f * LAYER_DP[src].sum() / LAYER_DP[cloud].sum())
        for i in lev[src]:
            M[c, i, :] = (1 - f) * (lev == i) + f * d
        for i in lev[cloud]:
            M[c, i, :] = 0.0; M[c, i, i] = 1 - s; M[c, i, i - 1] += s
    z = np.zeros(N)
    return M, conv, z, z.copy(), conv.astype(int), np.ones(N, bool)

TEACHER = teacher_flexpart if CFG["teacher"] == "flexpart" else teacher_standin
print("Teacher:", CFG["teacher"])
t0 = time.perf_counter()
M_all, conv_all, cbold_all, cbnew_all, iflag_all, valid_all = TEACHER(T_all, q_all)
print(f"Labelling: {time.perf_counter() - t0:.1f} s")

keep = valid_all
T_all, q_all, ps_all = T_all[keep], q_all[keep], ps_all[keep]
M_all, conv_all, cbold_all, cbnew_all = M_all[keep], conv_all[keep], cbold_all[keep], cbnew_all[keep]
if times_all is not None:
    times_all = times_all[keep]
print(f"Dropped {np.sum(~keep):,} columns (too short) | kept {len(T_all):,} | convective {conv_all.mean():.1%}")
print("Row sums all 1:", np.allclose(M_all.sum(axis=2), 1.0, atol=1e-4))

diag_all = parcel_diagnostics(T_all, q_all)

## 5. Features, splits and normalisation
Inputs: T, q (g/kg) on 32 levels, surface pressure, CAPE, CIN, LCL pressure and **cbmf_old**
→ `2 × 32 + 5 = 69` features.

In [ ]:
CBMF_SCALE = 100.0     # kg m⁻² s⁻¹ → ~O(1) numbers for training

def build_features(T, q, ps, diag, cb_old):
    return np.concatenate([
        T, q * 1000.0, ps[:, None],
        diag["cape"][:, None] / 1000.0, diag["cin"][:, None] / 100.0, diag["lcl_p"][:, None] / 1000.0,
        cb_old[:, None] * CBMF_SCALE,
    ], axis=1).astype(np.float32)

X_all = build_features(T_all, q_all, ps_all, diag_all, cbold_all)
CBOLD_COL = X_all.shape[1] - 1
y_all = conv_all.astype(np.float32)
N = len(X_all)

if times_all is not None:
    years = times_all.astype("datetime64[Y]").astype(int) + 1970
    uy = np.unique(years)
    if len(uy) >= 3:
        tr, va, te = np.isin(years, uy[:-2]), years == uy[-2], years == uy[-1]
    else:
        print("Fewer than 3 years — random split."); times_all = None
if times_all is None:
    perm = rng.permutation(N)
    tr, va, te = np.zeros(N, bool), np.zeros(N, bool), np.zeros(N, bool)
    tr[perm[: int(0.8 * N)]] = True; va[perm[int(0.8 * N): int(0.9 * N)]] = True; te[perm[int(0.9 * N):]] = True

X_mean = X_all[tr].mean(axis=0); X_std = X_all[tr].std(axis=0) + 1e-6
Xn = (X_all - X_mean) / X_std
print(f"train {tr.sum():,} | val {va.sum():,} | test {te.sum():,} | features {X_all.shape[1]}")
np.savez(os.path.join(OUT_DIR, "normalisation.npz"), mean=X_mean, std=X_std, p_levels=P_LEVELS)

## 6. The three models

In [ ]:
def mlp(d_in, hidden, d_out):
    layers, d = [], d_in
    for h in hidden:
        layers += [nn.Linear(d, h), nn.SiLU()]; d = h
    return nn.Sequential(*layers, nn.Linear(d, d_out))

class GateNet(nn.Module):
    def __init__(self, d_in):
        super().__init__(); self.net = mlp(d_in, [128, 128], 1)
    def forward(self, x):
        return self.net(x).squeeze(-1)

class RedistNet(nn.Module):
    def __init__(self, d_in, n=NLEV):
        super().__init__(); self.n = n; self.net = mlp(d_in, [256, 256, 256], n * n)
    def forward(self, x):
        return F.log_softmax(self.net(x).view(-1, self.n, self.n), dim=-1)

class CbmfNet(nn.Module):
    def __init__(self, d_in):
        super().__init__(); self.net = mlp(d_in, [64, 64], 1)
    def forward(self, x):                      # predicts cbmf_new × CBMF_SCALE (non-negative)
        return F.softplus(self.net(x).squeeze(-1))

W_LEV = torch.tensor(LAYER_W, dtype=torch.float32, device=DEVICE)

def weighted_kl(logM_hat, M):
    return ((torch.xlogy(M, M) - M * logM_hat).sum(dim=-1) * W_LEV).sum(dim=-1).mean()

d_in = X_all.shape[1]
gate, redist, cbnet = GateNet(d_in).to(DEVICE), RedistNet(d_in).to(DEVICE), CbmfNet(d_in).to(DEVICE)
npar = lambda m: sum(p.numel() for p in m.parameters())
print(f"Gate {npar(gate):,} | Redistribution {npar(redist):,} | Cbmf {npar(cbnet):,} parameters "
      f"(≈ {4 * (npar(gate) + npar(redist) + npar(cbnet)) / 1e6:.2f} MB)")

## 7. Training (best checkpoint kept, saved every epoch)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def loader(mask, shuffle=True):
    t = [torch.tensor(Xn[mask]), torch.tensor(y_all[mask]),
         torch.tensor(M_all[mask], dtype=torch.float32),
         torch.tensor(cbnew_all[mask] * CBMF_SCALE, dtype=torch.float32)]
    return DataLoader(TensorDataset(*t), batch_size=CFG["batch_size"], shuffle=shuffle)

def train(model, step_fn, train_dl, val_dl, epochs, name):
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    ckpt = os.path.join(OUT_DIR, f"{name}.pt"); best, hist = np.inf, []
    for ep in range(1, epochs + 1):
        model.train(); tl = []
        for b in train_dl:
            loss = step_fn(model, [x.to(DEVICE) for x in b])
            opt.zero_grad(); loss.backward(); opt.step(); tl.append(loss.item())
        model.eval()
        with torch.no_grad():
            vl = np.mean([step_fn(model, [x.to(DEVICE) for x in b]).item() for b in val_dl])
        sched.step(vl); hist.append((np.mean(tl), vl))
        if vl < best:
            best = vl; torch.save(model.state_dict(), ckpt)
        print(f"[{name}] epoch {ep:3d}  train {np.mean(tl):.5f}  val {vl:.5f}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return hist

pos_w = torch.tensor((1 - y_all[tr].mean()) / max(y_all[tr].mean(), 1e-6), device=DEVICE)
gate_step = lambda m, b: F.binary_cross_entropy_with_logits(m(b[0]), b[1], pos_weight=pos_w)
redist_step = lambda m, b: weighted_kl(m(b[0]), b[2])
cbmf_step = lambda m, b: F.mse_loss(m(b[0]), b[3])

t0 = time.perf_counter()
gate_hist = train(gate, gate_step, loader(tr), loader(va, False), CFG["gate_epochs"], "gate")
redist_hist = train(redist, redist_step, loader(tr & conv_all), loader(va & conv_all, False), CFG["redist_epochs"], "redist")
cbmf_hist = train(cbnet, cbmf_step, loader(tr & conv_all), loader(va & conv_all, False), CFG["cbmf_epochs"], "cbmf")
print(f"Training time: {(time.perf_counter() - t0) / 60:.1f} min")

## 8. Validation on the test set

In [ ]:
@torch.no_grad()
def predict(xn):
    x = torch.tensor(xn, device=DEVICE)
    prob = torch.sigmoid(gate(x)).cpu().numpy()
    M_hat = redist(x).exp().cpu().numpy()
    cb_hat = cbnet(x).cpu().numpy() / CBMF_SCALE
    return prob, M_hat, cb_hat

prob_te, Mhat_te, cbhat_te = predict(Xn[te])
y_te, M_te, cbnew_te = y_all[te].astype(bool), M_all[te], cbnew_all[te]
pred_te = prob_te > 0.5
Mfull_te = np.where(pred_te[:, None, None], Mhat_te, np.eye(NLEV)[None])

tp = (pred_te & y_te).sum(); fp = (pred_te & ~y_te).sum(); fn = (~pred_te & y_te).sum()
f1 = 2 * tp / max(2 * tp + fp + fn, 1)
mae_conv = np.abs(Mhat_te[y_te] - M_te[y_te]).mean()
mae_full = np.abs(Mfull_te - M_te).mean()

def upward_flux(M):
    m = LAYER_DP[None, :, None]
    return np.stack([(m[:, :k + 1] * M[:, :k + 1, k + 1:]).sum(axis=(1, 2)) for k in range(NLEV - 1)], axis=1)
flux_t, flux_m = upward_flux(M_te[y_te]).mean(0), upward_flux(Mhat_te[y_te]).mean(0)
sig = flux_t > 0.01 * flux_t.max()
flux_max_rel = (np.abs(flux_m - flux_t)[sig] / flux_t[sig]).max()

def final_cdf(M):
    return np.cumsum((LAYER_W[None, :, None] * M).sum(axis=1), axis=1)
w1 = (np.abs(final_cdf(Mhat_te[y_te]) - final_cdf(M_te[y_te])) * LAYER_DP[None, :]).sum(axis=1)

cb_t, cb_m = cbnew_te[y_te], cbhat_te[y_te]
_var = np.sum((cb_t - cb_t.mean()) ** 2)
cb_r2 = 1 - np.sum((cb_m - cb_t) ** 2) / _var if _var > 1e-12 else float("nan")  # nan for the stand-in (no memory)

# Speed on CPU: one teacher call per column vs one emulator pass
cpu = {k: type(m)(d_in) for k, m in [("gate", gate), ("redist", redist), ("cbmf", cbnet)]}
for k, m in [("gate", gate), ("redist", redist), ("cbmf", cbnet)]:
    cpu[k].load_state_dict(m.state_dict()); cpu[k].eval()
nb = min(2000, te.sum()); xb = torch.tensor(Xn[te][:nb])
t0 = time.perf_counter()
with torch.no_grad():
    cpu["gate"](xb); cpu["redist"](xb).exp(); cpu["cbmf"](xb)
t_model = (time.perf_counter() - t0) / nb
Tt, qt, cbt = T_all[te][:nb], q_all[te][:nb], cbold_all[te][:nb]
t0 = time.perf_counter()
if CFG["teacher"] == "flexpart":
    for c in range(nb):
        flexpart_column(Tt[c], qt[c], cbt[c])
else:
    teacher_standin(Tt, qt)
t_teach = (time.perf_counter() - t0) / nb

results = {
    "teacher": CFG["teacher"],
    "gate_F1": float(f1), "gate_TP_FP_FN": [int(tp), int(fp), int(fn)],
    "matrix_MAE_convective": float(mae_conv), "matrix_MAE_full_pipeline": float(mae_full),
    "mass_flux_max_relative_error": float(flux_max_rel),
    "particle_W1_hPa_median": float(np.median(w1)), "particle_W1_hPa_p95": float(np.percentile(w1, 95)),
    "cbmf_R2": float(cb_r2),
    "row_sums_equal_1": bool(np.allclose(Mfull_te.sum(axis=2), 1.0, atol=1e-5)),
    "time_per_column_teacher_us": t_teach * 1e6, "time_per_column_model_us": t_model * 1e6,
    "speedup": t_teach / t_model,
}
print(json.dumps(results, indent=2))
with open(os.path.join(OUT_DIR, "validation_results.json"), "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4.5))
ax[0].plot([h[0] for h in redist_hist], label="train"); ax[0].plot([h[1] for h in redist_hist], label="validation")
ax[0].set(title="Redistribution net loss", xlabel="epoch", yscale="log"); ax[0].legend()
ax[1].plot(flux_t, P_LEVELS[:-1], "k-", label="FLEXPART" if CFG["teacher"] == "flexpart" else "teacher")
ax[1].plot(flux_m, P_LEVELS[:-1], "C1--", label="emulator")
ax[1].invert_yaxis(); ax[1].set_yscale("log")
ax[1].set(title="Mean upward mass flux", xlabel="hPa of air crossing level", ylabel="pressure (hPa)"); ax[1].legend()
ax[2].hist(w1, bins=40); ax[2].set(title="Particle test (Wasserstein)", xlabel="hPa", ylabel="columns")
ax[3].scatter(cb_t, cb_m, s=4, alpha=0.4); lim = max(cb_t.max(), cb_m.max())
ax[3].plot([0, lim], [0, lim], "k--", lw=1); ax[3].set(title=f"cbmf_new (R² = {cb_r2:.3f})",
                                                     xlabel="FLEXPART (kg m⁻² s⁻¹)", ylabel="emulator")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "validation_summary.png"), dpi=130); plt.show()

idx = np.where(y_te)[0][np.argmax(diag_all["cape"][te][y_te])]
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, mat, ttl in [(ax[0], M_te[idx], "FLEXPART M"), (ax[1], Mhat_te[idx], "Emulator M̂"),
                    (ax[2], Mhat_te[idx] - M_te[idx], "Difference")]:
    d = ttl == "Difference"
    im = a.imshow(mat, origin="lower", cmap="RdBu_r" if d else "viridis",
                  vmin=-0.1 if d else 0, vmax=0.1 if d else mat.max())
    a.set(title=f"{ttl} (deepest test column)", xlabel="to level j", ylabel="from level i"); plt.colorbar(im, ax=a)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "example_matrix.png"), dpi=130); plt.show()

## 9. Export to TorchScript (for FLEXPART coupling in Phase 2)
Input: the 69 **raw** features (normalisation built in). Output: convection probability, the full
redistribution matrix (identity where no convection) and `cbmf_new` (unchanged `cbmf_old` where
no convection, matching FLEXPART's `calcmatrix` rule).

In [ ]:
class ConvectionEmulator(nn.Module):
    def __init__(self, gate, redist, cbnet, mean, std, cbold_col: int):
        super().__init__()
        self.gate, self.redist, self.cbnet = gate, redist, cbnet
        self.register_buffer("mean", torch.tensor(mean)); self.register_buffer("std", torch.tensor(std))
        self.register_buffer("eye", torch.eye(redist.n)); self.cbold_col = cbold_col
    def forward(self, x_raw):
        x = (x_raw - self.mean) / self.std
        prob = torch.sigmoid(self.gate(x)); on = prob > 0.5
        M = torch.where(on[:, None, None], self.redist(x).exp(), self.eye.expand(x.shape[0], -1, -1))
        cb_old = x_raw[:, self.cbold_col] / CBMF_SCALE
        cb_new = torch.where(on, self.cbnet(x) / CBMF_SCALE, cb_old)
        return prob, M, cb_new

emu = ConvectionEmulator(cpu["gate"], cpu["redist"], cpu["cbmf"], X_mean, X_std, CBOLD_COL).eval()
with torch.no_grad():
    scripted = torch.jit.trace(emu, torch.tensor(X_all[:4]))
ts_path = os.path.join(OUT_DIR, "convection_emulator_torchscript.pt")
scripted.save(ts_path)
with torch.no_grad():
    _, M_chk, _ = torch.jit.load(ts_path)(torch.tensor(X_all[te][:256]))
print("TorchScript saved:", ts_path)
print("Reloaded model matches:", np.allclose(M_chk.numpy(), Mfull_te[:256], atol=1e-5))

## Next steps
- Switch to ERA5 columns (at least 3 years, including levels up to 10 hPa or above) for a split by time.
- Add a held-out region (e.g. tropical Atlantic) and compare its error to the training regions.
- Phase 2: call the TorchScript model from FLEXPART's Fortran code through FTorch.